# Prepare long strips that have already been processed, for viewer tool

In [4]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [5]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,212.0,2025-07-25,GHL_OldStriper_20250725T1509,722Valves0.325to0.4,RG_7-22_12,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
218,213.0,2025-07-25,GHL_OldStriper_20250725T1514,722Valves0.325to0.4,RG_7-22_18,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
219,214.0,2025-07-25,GHL_OldStriper_20250725T1528,722Valves0.325to0.4,RG_7-22_10,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
220,215.0,2025-07-25,GHL_OldStriper_20250725T1532,722Valves0.325to0.4,RG_7-22_9,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN


In [7]:
#%% Filter The Tests To View
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     (dftests['Test date']=='2025-05-05')
# ]
dfmasks = [
    #dftests['Test date']=='2025-06-05',
    #(dftests['Test date']=='2025-06-11') | (dftests['Test date']=='2025-06-12'),
    #(dftests['Test date']=='2025-06-05') | (dftests['Test date']=='2025-06-11') | (dftests['Test date']=='2025-06-12'),
    #(dftests['Test date']>='2025-06-05') & (dftests['Test date']<'2025-07-25')
    (dftests['Test date']>='2025-06-05')
    #dftests['Test name']!='GHL_pyapp_20250512T',
    #dftests['ProcessingNotes'].str.startswith('done,2'),
    #~dftests['Batch'].str.contains('Valve')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
74,69.0,2025-06-05,GHL_pyapp_20250605T1211,Jun5 0.325-0.349mg/mm bag ylw,BluePenES0323#16 0.335,0.335,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,2,processing decent",redo fail2
75,70.0,2025-06-05,GHL_pyapp_20250605T1214,Jun5 0.325-0.349mg/mm bag ylw,ES0403#8 0.326,0.326,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1,processing decent",redo fail2
76,71.0,2025-06-05,GHL_pyapp_20250605T1218,Jun5 0.325-0.349mg/mm bag ylw,ES0402#11 0.342,0.342,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1,bad results processing",NaN
77,72.0,2025-06-05,GHL_pyapp_20250605T1225,Jun5 0.325-0.349mg/mm bag ylw,ES0403#6 0.331,0.331,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1,processing decent",NaN
78,73.0,2025-06-05,GHL_pyapp_20250605T1228,Jun5 0.325-0.349mg/mm bag ylw,ES0403#19 0.329,0.329,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1,processing decent",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,212.0,2025-07-25,GHL_OldStriper_20250725T1509,722Valves0.325to0.4,RG_7-22_12,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
218,213.0,2025-07-25,GHL_OldStriper_20250725T1514,722Valves0.325to0.4,RG_7-22_18,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
219,214.0,2025-07-25,GHL_OldStriper_20250725T1528,722Valves0.325to0.4,RG_7-22_10,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN
220,215.0,2025-07-25,GHL_OldStriper_20250725T1532,722Valves0.325to0.4,RG_7-22_9,NaN,Vacuum,9.0,3.5,1.998125,...,297.3,132.5,164.8,"Default (Medium sensitivity, 48 kHz)",2.0,1.6,0.18,3.45,"done 7/25, 10.6 hours for 38 strips",NaN


In [8]:
#%% Load OCT study information
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

folder_temp = Path(r'D:\TEMP')

In [9]:
# Instantiate OCT_Study_Folder objects for each 
octstudies = [];
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);

STUDY: GHL_pyapp_20250605T1211
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1214
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1218
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1225
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1228
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

# Parse Through The OCTSTUDIES

In [10]:
#%% Load OCTSTUDY object, and read it's summary / results data
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    pp.pprint(octstudy.resultsCheck())
    doWeRunTheNotebook = any([v is False for k,v in octstudy.resultsCheck().items()])
    print('Run?',doWeRunTheNotebook)
    if(not doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

print('~~~~~~~');
print('We will analyze data for {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run]);

~~~~~~~
GHL_pyapp_20250605T1211
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1214
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1218
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1225
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1228
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1233
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
GHL_pyapp_20250605T1301
{   'along_strip_data_extracted': ['/dfstepA

# 1. Load all Data

In [11]:
#octstudies_to_run = [octstudies_to_run[0]]
dfs = [];
for idx,octstudy in enumerate(octstudies_to_run):
    print(octstudy.name)

    #octstudy.load_previously_saved_merged_volume();

    data_extracted = octstudy.load_data_extracted();
    dfstep = octstudy.load_data_extracted_along_strip();
    
    dfstep['octstudy'] = octstudy;

    dfs.append(dfstep)
dfsteps = pd.concat(dfs,keys=[x.name for x in octstudies_to_run]);
del dfs;

GHL_pyapp_20250605T1211
Loading /dfstepA
GHL_pyapp_20250605T1214
Loading /dfstepA
GHL_pyapp_20250605T1218
Loading /dfstepA
GHL_pyapp_20250605T1225
Loading /dfstepA
GHL_pyapp_20250605T1228
Loading /dfstepA
GHL_pyapp_20250605T1233
Loading /dfstepA
GHL_pyapp_20250605T1301
Loading /dfstepA
GHL_pyapp_20250605T1326
Loading /dfstepA
GHL_pyapp_20250605T1330
Loading /dfstepA
GHL_pyapp_20250605T1333
Loading /dfstepA
GHL_pyapp_20250605T1336
Loading /dfstepA
GHL_pyapp_20250605T1340
Loading /dfstepA
GHL_pyapp_20250605T1344
Loading /dfstepA
GHL_pyapp_20250605T1347
Loading /dfstepA
GHL_pyapp_20250605T1350
Loading /dfstepA
GHL_pyapp_20250605T1402
Loading /dfstepA
GHL_pyapp_20250605T1407
Loading /dfstepA
GHL_pyapp_20250605T1410
Loading /dfstepA
GHL_pyapp_20250605T1414
Loading /dfstepA
GHL_pyapp_20250605T1417
Loading /dfstepA
GHL_pyapp_20250605T1422
Loading /dfstepA
GHL_pyapp_20250605T1425
Loading /dfstepA
GHL_pyapp_20250605T1428
Loading /dfstepA
GHL_pyapp_20250605T1431
Loading /dfstepA
GHL_pyapp_202506

In [12]:
import gc
gc.collect()

0

In [13]:
dfsteps

pixel_depth_strip_top_sum_threshold  \
                             slice                                        
GHL_pyapp_20250605T1211      5                                    30.24   
                             10                                   32.16   
                             15                                   33.40   
                             20                                   35.56   
                             25                                   34.44   
...                                                                 ...   
GHL_OldStriper_20250725T1519 8010                                 53.16   
                             8015                                 55.56   
                             8020                                 57.64   
                             8025                                 59.68   
                             8030                                 60.40   

                                    pixel_depth_strip_top  \
                             slice                          
GHL_pyapp_20250605T1211      5                        323   
                             10                       322   
                             15                       322   
                             20                       322   
                             25                       321   
...                                                   ...   
GHL_OldStriper_20250725T1519 8010                     203   
                             8015                     204   
                             8020                     204   
                             8025                     205   
                             8030                     205   

                                    pixel_depth_strip_bot  \
                             slice                          
GHL_pyapp_20250605T1211      5                        369   
                             10                       369   
                             15                       370   
                             20                       368   
                             25                       370   
...                                                   ...   
GHL_OldStriper_20250725T1519 8010                     247   
                             8015                     246   
                             8020                     247   
                             8025                     247   
                             8030                     247   

                                    pixel_depth_wax_center  \
                             slice                           
GHL_pyapp_20250605T1211      5                         357   
                             10                        356   
                             15                        349   
                             20                        350   
                             25                        368   
...                                                    ...   
GHL_OldStriper_20250725T1519 8010                      228   
                             8015                      228   
                             8020                      232   
                             8025                      232   
                             8030                      232   

                                                     px_wax_transverse_edges  \
                             slice                                             
GHL_pyapp_20250605T1211      5      (56.035470941883766, 115.66230215827338)   
                             10         (54.3269592476489, 116.359941089838)   
                             15     (49.520063191153234, 118.82182061579653)   
                             20     (50.178378378378376, 119.19511904761904)   
                             25      (42.08465608465608, 118.88260869565218)   
...                                                                      ...   
GHL_OldStriper_20250725T1

In [17]:
dffilt['Batch'].unique()

array(['Jun5 0.325-0.349mg/mm bag ylw',
       'Jun5 0.375-0.400mg/mm bag blu crossouts',
       'Jun11InSpecSmallZiploc', 'July11WaxrobotStrips',
       'July17n18Oldstriper', '722Valves0.325to0.4'], dtype=object)

## 2. Save

In [18]:
cols_ignore = ['testseg_otsu_thresholds','testseg_otsu_regions','seg_thresh_sauvola_cutoff','pksA','pksB','pksB2','wax_segmentation']

df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]
#df.to_hdf(folder_temp/'oct_results_summary_2025_06_17_allbathces.hdf',key='table');
#df.to_hdf(folder_temp/'oct_results_summary_2025_07_03_allbatches.hdf',key='table');
df.to_hdf(folder_temp/'oct_results_summary_2025_07_25b_allbatches.hdf',key='table');

C:\Users\SimonGhionea\AppData\Local\Temp\ipykernel_143048\2241103483.py:6: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block2_values] [items->Index(['px_wax_transverse_edges', 'wax_top_seed_candidate_px', 'octstudy'], dtype='object')]

  df.to_hdf(folder_temp/'oct_results_summary_2025_07_25b_allbatches.hdf',key='table');
